# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I build the feature vector the same way the reference pipeline does
(`scripts/01_prepare_features.py` + the feature lists in `scripts/ml_utils.py`), so
my leakage tests measure the real model design, not a straw man:

- **Fills:** numeric blanks → 0, categorical blanks → `"unknown"` — the reference
  prep's choices, kept deliberately so results stay comparable (section 2 notes
  where this is a compromise).
- **Engineered:** `log1p` of the four heavy-tailed 90-day totals
  (`impressions`, `clicks`, `sessions`, `ai_sessions`), plus the label
  `is_declining_label = (trend_direction == "down")`.
- **Model matrix:** the 26 reference features — 18 numeric passed through, 8
  categorical one-hot encoded. IDs are never in the matrix; they are kept aside
  for the client-holdout split.

The prediction moment this design serves: **at export/scoring time, rank a client's
existing pages for refresh review.** Every feature must be knowable at that moment —
section 3 attacks exactly that.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Find the repo root by walking up from wherever this notebook was opened.
ROOT = Path.cwd()
while not (ROOT / "data/raw/content_refresh_anonymized.csv").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))

RAW = ROOT / "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(RAW)
print("Loaded:", RAW.relative_to(ROOT), "| shape:", df.shape)

from ml_utils import (MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES,
                      NUMERIC_COLUMNS, CATEGORICAL_COLUMNS)

work = df.copy()
# Reference prep (scripts/01_prepare_features.py): coerce, fill, filter, dedupe.
for c in NUMERIC_COLUMNS:
    work[c] = pd.to_numeric(work[c], errors="coerce")
for c in CATEGORICAL_COLUMNS:
    work[c] = work[c].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})
work[list(NUMERIC_COLUMNS)] = (work[list(NUMERIC_COLUMNS)]
                               .replace([np.inf, -np.inf], np.nan).fillna(0))
work = work[(work["impressions_90d"] > 0) & (work["content_age_days"] >= 90)]
work = work.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

# Engineered features + the label.
work["is_declining_label"] = work["trend_direction"].str.lower().eq("down").astype(int)
for new, raw in [("log_impressions_90d", "impressions_90d"), ("log_clicks_90d", "clicks_90d"),
                 ("log_sessions_90d", "sessions_90d"), ("log_ai_sessions_90d", "ai_sessions_90d")]:
    work[new] = np.log1p(work[raw])

FEATURES = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
print("feature vector rows:", len(work), "| label rate:", round(work["is_declining_label"].mean(), 4))
print("model features:", len(FEATURES), "=", len(MODEL_NUMERIC_FEATURES), "numeric +",
      len(MODEL_CATEGORICAL_FEATURES), "categorical")

# One-hot encode the categoricals to get the final model matrix width.
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"),
                          MODEL_CATEGORICAL_FEATURES)], remainder="passthrough")
X = pre.fit_transform(work[FEATURES])
print("model matrix after one-hot:", X.shape)
print("NaNs in matrix:", int(np.isnan(X).sum()), "| infs:", int(np.isinf(X).sum()))


Loaded: data/raw/content_refresh_anonymized.csv | shape: (30000, 44)


feature vector rows: 30000 | label rate: 0.5421
model features: 26 = 18 numeric + 8 categorical


model matrix after one-hot: (30000, 52)
NaNs in matrix: 0 | infs: 0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Grouped notes (individual columns behave like their group):

| Feature group | Meaning | Missing → handling | Available when? |
|---|---|---|---|
| Keyword context: `search_volume`, `competition`, `cpc`, `competition_level` | Demand and difficulty of the page's target keyword | Blank for all feedly rows and ~8% overall → filled 0 / `"unknown"`. **Compromise I am flagging, not hiding:** filling 0 makes "no keyword data" look like "zero-volume keyword". A missingness indicator would be cleaner; I keep the reference fill so my numbers stay comparable to the reference pipeline. | Known from content metadata — exists before scoring. |
| Content properties: `word_count`, `char_count`, `word_count_tier`, `content_type`, `main_intent`, `age_tier`, `freshness_tier`, `content_age_days`, `days_since_last_update` | What the page is and how old / stale it is | `word_count`/`char_count` blank for 25.7% (unmeasured pages) → 0 / tier `"unknown"` | Properties of the page itself — exist before scoring. |
| Activity totals (as logs): `log_impressions_90d`, `log_clicks_90d`, `log_sessions_90d`, `log_ai_sessions_90d`, plus `days_with_impressions`, `days_with_sessions` | How much observed traffic the page had over the trailing 90 days | Never blank in this slice; log1p tames the heavy tails (impressions: median 731, max 517,715) | Measured over the trailing 90 days up to export — available at scoring time. **But see §3, attack 3:** that window contains the label's own comparison windows. |
| Rates: `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `impression_tier`, `position_tier` | Derived ×100 percentages and threshold tiers from the same 90 days | `avg_position = 0` means *unmeasured* (1,205 rows) yet is filled as the number 0 — the model can learn "0 = invisible page", which is honest signal, but nobody should read it as a rank. `scroll_rate` blank when pageviews = 0 → 0. | Same window caveat as the activity totals. |

The honest summary: every feature is knowable at scoring time for a **snapshot
ranking**. None of them is knowable *before the label window starts* — that
distinction is the whole of attack 3 below.


In [2]:
# --- Missingness of the features BEFORE fills, measured on the raw export ---
# (the 4 log features are engineered from the 90-day totals, so I audit their
#  source columns - a log feature is blank exactly when its source total is)
LOG_SOURCES = ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]
raw_feature_cols = [c for c in FEATURES if not c.startswith("log_")] + LOG_SOURCES
raw_miss = (df[raw_feature_cols].isna().mean() * 100).round(1).sort_values(ascending=False)
print("Feature missingness in the raw export (%, log sources included):")
print(raw_miss[raw_miss > 0])
print("feature columns with no missing values:", int((raw_miss == 0).sum()),
      "of", len(raw_feature_cols))

# --- After the reference fills, the matrix must be complete ---
print()
print("NaNs left in work[FEATURES]:", int(work[FEATURES].isna().sum().sum()))

# --- avg_position: how much of it is the 'unmeasured' zero? ---
print("avg_position == 0 rows (unmeasured, filled as 0):",
      int((work["avg_position"] == 0).sum()), "=",
      round(100 * (work["avg_position"] == 0).mean(), 1), "% of rows")

# --- Spot-check an engineered feature ---
row = work.iloc[0]
print("spot check: log1p(impressions_90d) =",
      round(float(np.log1p(row["impressions_90d"])), 4),
      "| stored log_impressions_90d =", round(float(row["log_impressions_90d"]), 4))


Feature missingness in the raw export (%, log sources included):
char_count           25.7
word_count           25.7
word_count_tier      25.7
competition_level     8.7
competition           8.2
search_volume         8.2
cpc                   8.2
main_intent           7.9
scroll_rate           0.4
dtype: float64
feature columns with no missing values: 17 of 26

NaNs left in work[FEATURES]: 0
avg_position == 0 rows (unmeasured, filled as 0): 1205 = 4.0 % of rows
spot check: log1p(impressions_90d) = 8.2438 | stored log_impressions_90d = 8.2438


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Three attacks. Base rate printed first, as always: **54.2%** of rows are declining,
so a majority-class guess scores 54.2% accuracy and any ranking metric must be read
against that. All model tests use one random forest (300 trees, seed 42) and the
**client-holdout split built the same way as `scripts/03_train_model.py`** — 6 of 32
clients held out entirely (2,325 test rows; their label rate is 39.1%, not 54.2%,
which is itself a lesson in why per-split base rates matter).

**Attack 1 — label-derived columns (the confession).** The label is *computed from*
`trend_direction`, and `trend_pct` is the same comparison as a number. Test: add
`trend_pct` to the clean features, same split, same model.
Clean: **ROC AUC 0.740, Precision@50 0.86**. With `trend_pct`: **ROC AUC 1.000,
Precision@50 1.00**. And `-trend_pct` *alone*, no model at all, scores AUC ≈ 1.000 —
the feature doesn't predict the label, it *is* the label (below −20% → 99.7%
declining; ≥ −20% → 0.0%). Verdict: `trend_direction` and `trend_pct` are excluded,
permanently. Any future feature whose single-feature AUC looks like this gets the
same treatment.

**Attack 2 — the split itself.** Same clean features, but a random row split
instead of client-holdout: **AUC 0.776, Precision@50 0.98** vs the honest
0.740 / 0.86. Pages from one client share hidden character, so a random split lets
the model partly memorise clients and fake skill. The gap between the two numbers
is the size of that fake skill. All numbers I report use client-holdout.

**Attack 3 — window overlap (the subtle one, disclosed, not hidden).** The 90-day
feature aggregates *contain* the label's last-30/prev-30 windows. This is not
label-copying — no feature is computed from `trend_direction` — but it does mean
the model can only ever answer "which pages are observed declining **in this
snapshot**", which is exactly the refresh-prioritisation task. It must never be
sold as forecasting a future decline. The label's direct ingredients (the six
last/prev-30-day counts) are excluded from the features for the same reason. On
the full warehouse release, a forward-looking label would only allow
`*_prev30`-style windows as features.

**Product-flag check.** There are no product decision flags in this export to leak
(`DATA_USE.md`: `health_score`, `needs_ctr_fix`, … are intentionally absent) — I
verified the column inventory rather than assuming it. `provider_used` /
`model_used` are excluded as process metadata (section 4).


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import sys
sys.path.insert(0, str(ROOT / "scripts"))
from ml_utils import precision_at_k

y = work["is_declining_label"]
print("base rate (share declining):", round(y.mean(), 4))

# Client-holdout split, same construction as scripts/03_train_model.py (seed 42):
# shuffle the 32 clients, hold out 20% of CLIENTS (not rows).
rng = np.random.default_rng(42)
clients = work["client_id"].drop_duplicates().to_numpy()
test_clients = set(rng.permutation(clients)[:max(1, int(round(len(clients) * 0.2)))])
te_mask = work["client_id"].isin(test_clients).to_numpy()
tr_idx, te_idx = np.where(~te_mask)[0], np.where(te_mask)[0]
print("holdout clients:", len(test_clients), "| test rows:", len(te_idx),
      "| test label rate:", round(y.iloc[te_idx].mean(), 3))

def run(feat_num, feat_cat, tr, te):
    X = work[feat_num + feat_cat]
    p = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), feat_cat)],
                          remainder="passthrough")
    m = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    m.fit(p.fit_transform(X.iloc[tr]), y.iloc[tr])
    prob = m.predict_proba(p.transform(X.iloc[te]))[:, 1]
    return round(roc_auc_score(y.iloc[te], prob), 3), round(precision_at_k(y.iloc[te], prob, 50), 2)

print()
print("ATTACK 1  clean features, client-holdout:      ",
      run(MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, tr_idx, te_idx))
print("ATTACK 1  + trend_pct (LEAK), client-holdout:  ",
      run(MODEL_NUMERIC_FEATURES + ["trend_pct"], MODEL_CATEGORICAL_FEATURES, tr_idx, te_idx))
print("ATTACK 1  -trend_pct alone as the score: AUC =",
      round(roc_auc_score(y, -work["trend_pct"]), 4))
print()
r_tr, r_te = train_test_split(np.arange(len(work)), test_size=0.2, random_state=42)
print("ATTACK 2  clean features, RANDOM row split:    ",
      run(MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, r_tr, r_te))

print()
print("Why trend_pct confesses: label rate by trend_pct band")
band = pd.cut(work["trend_pct"], [-1e9, -20, 0, 20, 1e9],
              labels=["< -20", "-20..0", "0..20", "> 20"])
print(work.groupby(band, observed=True)["is_declining_label"].agg(["mean", "count"]).round(3))


base rate (share declining): 0.5421
holdout clients: 6 | test rows: 2325 | test label rate: 0.391



ATTACK 1  clean features, client-holdout:       (0.74, 0.86)


ATTACK 1  + trend_pct (LEAK), client-holdout:   (1.0, 1.0)
ATTACK 1  -trend_pct alone as the score: AUC = 1.0



ATTACK 2  clean features, RANDOM row split:     (0.776, 0.98)

Why trend_pct confesses: label rate by trend_pct band


            mean  count
trend_pct              
< -20      0.997  16313
-20..0     0.000   7233
0..20      0.000   2068
> 20       0.000   4386


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded field | Why |
|---|---|
| `trend_direction` | It **is** the label — `is_declining_label` is computed from it. Using it is reading the answer. |
| `trend_pct` | The label's continuous sibling: same last-30 vs prev-30 comparison. Alone it scores AUC ≈ 1.000 (§3, attack 1). |
| `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` | The label's numerator window — half of the comparison that defines the trend. |
| `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` | The label's denominator window — the other half. Together with the row above, these six are the label's ingredients. |
| `content_id` | Pseudonymous ID. Grouping/joins only (privacy rule), and a unique-per-row column can only memorise. |
| `client_id` | Pseudonymous ID. Used for the client-holdout split — the one job IDs are allowed — never as a feature. |
| `provider_used` | Which LLM generated the page is process metadata, not a page property; also 71.5% missing. The dictionary marks it "not a model feature". |
| `model_used` | Same reasoning as `provider_used`; 19.1% missing. |
| `age_tier_order` | Numeric twin of `age_tier` — the same information twice. |
| `char_count_tier` | Unused twin of the length tiers; `char_count` + `word_count_tier` already carry the signal. |
| `pageviews_90d`, `users_90d`, `engaged_sessions_90d`, `scroll_events_90d` | Raw totals superseded by their rates (`engagement_rate`, `scroll_rate`) in the reference feature set — keeping both double-counts one signal. |
| `has_clicks`, `has_ai_sessions`, `measurable_opportunity` | Prep-step helper columns, not part of the reference model lists; left out so my numbers stay comparable with the reference pipeline. |

The code below proves the exclusion is real in the built matrix, not just promised
in this table.


In [4]:
EXCLUDED = ["trend_direction", "trend_pct",
            "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
            "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
            "content_id", "client_id", "provider_used", "model_used",
            "age_tier_order", "char_count_tier",
            "pageviews_90d", "users_90d", "engaged_sessions_90d", "scroll_events_90d",
            "has_clicks", "has_ai_sessions", "measurable_opportunity"]

leaked = [c for c in EXCLUDED if c in FEATURES]
print("excluded fields present in the feature list:", leaked)
assert not leaked, "an excluded field slipped into the features!"
assert "content_id" not in FEATURES and "client_id" not in FEATURES
print("final feature count:", len(FEATURES))
print("features:", FEATURES)
print()
print("All leakage-hunt assertions passed: the matrix contains no label-derived,")
print("label-ingredient, ID, or process-metadata columns.")


excluded fields present in the feature list: []
final feature count: 26
features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']

All leakage-hunt assertions passed: the matrix contains no label-derived,
label-ingredient, ID, or process-metadata columns.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.